In [1]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np
import pandas as pd
import random

In [2]:
df = pd.read_csv("../out/coefficients.csv")
columns = df.columns.tolist()

In [3]:
random.seed(89293)
features = sorted(random.sample(columns, 48), key=int)
target = sorted(list(set(columns) - set(features)), key=int)

In [146]:
X = df[features]
y = df[target]

X_standard_scaled = pd.DataFrame(StandardScaler().fit_transform(X))
X_min_max_scaled = pd.DataFrame(MinMaxScaler().fit_transform(X))

In [185]:
from joblib import Parallel, delayed, parallel_backend

from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.base import BaseEstimator, RegressorMixin


class MultiTargetWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, base_model, n_jobs=-1):
        self.base_model = base_model
        self.models = []
        self.n_jobs = n_jobs

    def fit(self, X, y):
        with parallel_backend("threading", n_jobs=self.n_jobs):
            self.models = Parallel()(
                delayed(self._fit_single_target)(X, y.iloc[:, i])
                for i in range(y.shape[1])
            )

    def _fit_single_target(self, X, y):
        model = self.base_model.__class__(**self.base_model.get_params())
        model.fit(X, y)
        return model

    def predict(self, X):
        with parallel_backend("threading", n_jobs=self.n_jobs):
            predictions = Parallel()(delayed(model.predict)(X) for model in self.models)
        return np.vstack(predictions).T


def fit_predict(
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    handle_value_error,
    criterion=mean_squared_error,
):
    try:
        model.fit(X_train, y_train)
        predictions = np.round(model.predict(X_test))
        return criterion(y_test, predictions)
    except ValueError as e:
        if handle_value_error:
            return float("inf")
        else:
            raise e

In [6]:
import optuna
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error

# Models
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [172]:
baseline_with_zeros = np.zeros(shape=y_test.values.shape)
print(
    f"First baseline (predict all coefficients with 0):"
    f"\nMSE = {mean_squared_error(y_test, baseline_with_zeros)}"
    f"\nMAPE = {mean_absolute_percentage_error(y_test, baseline_with_zeros)}"
)

First baseline (predict all coefficients with 0):
MSE = 169.73240439381613
MAPE = 0.13247558991049632


In [182]:
def cross_validation(
    regressor,
    n_splits,
    handle_value_error=True,
    scale=None,
    criterion=mean_squared_error,
) -> float:
    selected_X = (
        X_standard_scaled
        if scale == "Standard"
        else X_min_max_scaled
        if scale == "MinMax"
        else X
    )
    results = Parallel(n_jobs=-1)(
        delayed(fit_predict)(
            model=regressor,
            X_train=selected_X.loc[train],
            y_train=y.loc[train],
            X_test=selected_X.loc[test],
            y_test=y.loc[test],
            handle_value_error=handle_value_error,
            criterion=criterion,
        )
        for train, test in KFold(n_splits=n_splits, shuffle=True).split(selected_X, y)
    )
    return sum(results) / n_splits

In [140]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import typing


class MLPNet(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_layer_sizes: typing.Tuple[int],
        output_size: int,
        activation: str,
    ):
        super(MLPNet, self).__init__()

        if len(hidden_layer_sizes) == 0:
            self.model = nn.Linear(input_size, output_size)
            return

        layers = [nn.Linear(input_size, hidden_layer_sizes[0])]
        for i in range(len(hidden_layer_sizes)):
            layers.append(MLPNet.get_activation_layer(activation))
            if i < len(hidden_layer_sizes) - 1:
                layers.append(
                    nn.Linear(hidden_layer_sizes[i], hidden_layer_sizes[i + 1])
                )
        layers.append(nn.Linear(hidden_layer_sizes[-1], output_size))

        self.model = nn.Sequential(*layers)

    @staticmethod
    def get_activation_layer(activation: str) -> nn.Module:
        match activation:
            case "identity":
                return nn.Identity()
            case "logistic":
                return nn.Sigmoid()
            case "tanh":
                return nn.Tanh()
            case "relu":
                return nn.ReLU()
            case _:
                raise ValueError(f"Unknown activation function: {activation}")

    def forward(self, x):
        return self.model(x)

    @staticmethod
    def get_device_name() -> str:
        if torch.cuda.is_available():
            return "cuda"
        if torch.backends.mps.is_available():
            return "mps"
        return "cpu"

    def fit(self, train_loader: DataLoader, epochs: int, learning_rate=0.0001):
        device = torch.device(MLPNet.get_device_name())
        self.to(device=device)

        optimizer = torch.optim.Adam(self.parameters(), learning_rate)
        criterion = nn.MSELoss()

        for _ in range(epochs):
            self.train()

            for batch, (inputs, labels) in enumerate(train_loader):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = self(inputs)
                loss = criterion(outputs, labels)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

    def predict(self, X: torch.tensor) -> torch.tensor:
        device = torch.device(MLPNet.get_device_name())
        X = X.to(device)
        return self(X)


class TorchModelWrapper:
    INPUT_TYPE = torch.float32

    def __init__(self, model: nn.Module, fit_params) -> None:
        self.model = model
        self.fit_params = fit_params

    def fit(self, X: pd.DataFrame, y: pd.DataFrame) -> None:
        train_loader = DataLoader(
            dataset=TensorDataset(*map(TorchModelWrapper.to_tensor, (X, y))),
            batch_size=64,
            shuffle=True,
        )
        self.model.fit(train_loader=train_loader, **self.fit_params)

    def predict(self, X: pd.DataFrame) -> np.array:
        X = TorchModelWrapper.to_tensor(X)
        prediction = self.model.predict(X).cpu().detach().numpy()
        return prediction

    @staticmethod
    def to_tensor(X: pd.DataFrame) -> torch.tensor:
        return torch.tensor(X.values, dtype=TorchModelWrapper.INPUT_TYPE)

In [79]:
fit_predict(
    model=TorchModelWrapper(
        model=MLPNet(
            input_size=48,
            hidden_layer_sizes=(64, 96, 32),
            output_size=16,
            activation="relu",
        ),
        fit_params={"epochs": 100},
    ),
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
)

128.11665988608624

In [141]:
from datetime import datetime


class Loger:
    TIMESTAMP_PADDING = 22 * " "

    def on_optimizer_generate(self, trial):
        print(
            f"[{self._get_timestamp()}] Start trial {trial.study.study_name}.{trial.number}:\n"
            f"{logger.TIMESTAMP_PADDING}Params: {trial.params}"
        )

    def on_finished_cross_validataion(self, trial, error):
        print(
            f"[{self._get_timestamp()}] Finished cross validation {trial.study.study_name}.{trial.number}: error = {error}"
        )

    def on_finished_trial(self, study, trial):
        self._log(
            f"Finished trial {study.study_name}.{trial.number}:\n"
            f"{logger.TIMESTAMP_PADDING}Value: {trial.value}\n"
            f"{logger.TIMESTAMP_PADDING}Params: {trial.params}"
        )

    def _log(self, message: str) -> None:
        print(f"[{self._get_timestamp()}] {message}")

    def _get_timestamp(self) -> str:
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


logger = Loger()

In [126]:
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import optuna


def optimize_svr(trial):
    svr_c = trial.suggest_float("svr_c", 0.01, 1.0)
    kernel = trial.suggest_categorical("kernel", ["linear", "poly", "rbf", "sigmoid"])
    degree = 3 if kernel == "poly" else trial.suggest_int("degree", 1, 5)
    gamma = (
        "scale"
        if kernel in ["rbf", "poly", "sigmoid"]
        else trial.suggest_categorical("gamma", ["scale", "auto"])
    )
    coef0 = (
        0.0 if kernel in ["poly", "sigmoid"] else trial.suggest_float("coef0", -1, 1)
    )
    shrinking = trial.suggest_categorical("shrinking", [False, True])
    model = SVR(
        C=svr_c,
        kernel=kernel,
        degree=degree,
        gamma=gamma,
        coef0=coef0,
        shrinking=shrinking,
    )
    return MultiTargetWrapper(model)


def optimize_linear_regression(trial):
    return LinearRegression()


def optimize_kneighbors_regressor(trial):
    n_neighbors = trial.suggest_int("n_neighbors", 1, 20)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    metric = trial.suggest_categorical(
        "metric",
        ["cityblock", "cosine", "euclidean", "l1", "l2", "manhattan", "nan_euclidean"],
    )
    model = KNeighborsRegressor(n_neighbors=n_neighbors, weights=weights, metric=metric)
    return model


def optimize_decision_tree_regressor(trial):
    splitter = trial.suggest_categorical("splitter", ["best", "random"])
    max_depth = (
        trial.suggest_int("max_depth", 1, 50)
        if trial.suggest_categorical("use max_depth", [False, True])
        else None
    )
    model = DecisionTreeRegressor(splitter=splitter, max_depth=max_depth)
    return model


def optimize_random_forest_regressor(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 50)
    model = RandomForestRegressor(n_estimators=n_estimators)
    return model


def suggest_unsupported_categorical(trial, name, choices):
    i = trial.suggest_categorical(name, list(range(len(choices))))
    return choices[i]


def optimize_mlp_regressor(trial):
    hidden_layer_sizes = suggest_unsupported_categorical(
        trial, "hidden_layer_sizes", [(32,), (64,), (64, 32), (128, 64, 32)]
    )
    activation = trial.suggest_categorical(
        "activation", ["identity", "logistic", "tanh", "relu"]
    )
    # solver = trial.suggest_categorical("solver", ["lbfgs", "sgd", "adam"])
    learning_rate = trial.suggest_float("alpha", 0.001, 0.1, log=True)
    model = MLPNet(
        input_size=len(X.columns),
        hidden_layer_sizes=hidden_layer_sizes,
        output_size=len(y.columns),
        activation=activation,
    )
    return TorchModelWrapper(
        model=model, fit_params={"epochs": 100, "learning_rate": learning_rate}
    )


def objective(trial: optuna.trial.Trial):
    algorithm = trial.suggest_categorical(
        "algorithm",
        [
            # "SVR",
            "LinearRegression",
            "KNeighborsRegressor",
            "DecisionTreeRegressor",
            "RandomForestRegressor",
            "MLPRegressor",
        ],
    )

    if algorithm == "SVR":
        regressor_getter = optimize_svr
    elif algorithm == "LinearRegression":
        regressor_getter = optimize_linear_regression
    elif algorithm == "KNeighborsRegressor":
        regressor_getter = optimize_kneighbors_regressor
    elif algorithm == "DecisionTreeRegressor":
        regressor_getter = optimize_decision_tree_regressor
    elif algorithm == "RandomForestRegressor":
        regressor_getter = optimize_random_forest_regressor
    elif algorithm == "MLPRegressor":
        regressor_getter = optimize_mlp_regressor
    else:
        raise ValueError(f"Unexpected algorithm: {algorithm}")

    scale = trial.suggest_categorical("scale", [None, "Standard", "MinMax"])
    regressor = regressor_getter(trial)
    logger.on_optimizer_generate(trial=trial)
    error = cross_validation(
        regressor=regressor,
        n_splits=3,
        scale=scale,
        handle_value_error=False,
    )
    logger.on_finished_cross_validataion(trial, error)
    return error

In [148]:
optuna.logging.set_verbosity(optuna.logging.WARN)
study = optuna.create_study(
    study_name=str(0),
    sampler=optuna.samplers.TPESampler(),
)
study.optimize(objective, n_trials=100, callbacks=[logger.on_finished_trial])

best_trial = study.best_trial
print(f"Best score: {best_trial.value}")
print(f"Best params: {best_trial.params}")

[2023-12-19 03:26:13] Start trial 0.0:
                      Params: {'algorithm': 'MLPRegressor', 'scale': 'MinMax', 'hidden_layer_sizes': 0, 'activation': 'tanh', 'alpha': 0.0057282325774150105}
[2023-12-19 03:28:46] Finished cross validation 0.0: error = 137.36971028645834
[2023-12-19 03:28:46] Finished trial 0.0:
                      Value: 137.36971028645834
                      Params: {'algorithm': 'MLPRegressor', 'scale': 'MinMax', 'hidden_layer_sizes': 0, 'activation': 'tanh', 'alpha': 0.0057282325774150105}
[2023-12-19 03:28:46] Start trial 0.1:
                      Params: {'algorithm': 'RandomForestRegressor', 'scale': None, 'n_estimators': 38}
[2023-12-19 03:29:03] Finished cross validation 0.1: error = 136.70028686523438
[2023-12-19 03:29:03] Finished trial 0.1:
                      Value: 136.70028686523438
                      Params: {'algorithm': 'RandomForestRegressor', 'scale': None, 'n_estimators': 38}
[2023-12-19 03:29:03] Start trial 0.2:
                   

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[2023-12-19 04:34:55] Finished cross validation 0.61: error = 108.64772542317708
[2023-12-19 04:34:55] Finished trial 0.61:
                      Value: 108.64772542317708
                      Params: {'algorithm': 'KNeighborsRegressor', 'scale': None, 'n_neighbors': 4, 'weights': 'distance', 'metric': 'l1'}
[2023-12-19 04:34:55] Start trial 0.62:
                      Params: {'algorithm': 'KNeighborsRegressor', 'scale': None, 'n_neighbors': 4, 'weights': 'distance', 'metric': 'l1'}
[2023-12-19 04:34:59] Finished cross validation 0.62: error = 108.27193196614583
[2023-12-19 04:34:59] Finished trial 0.62:
                      Value: 108.27193196614583
                      Params: {'algorithm': 'KNeighborsRegressor', 'scale': None, 'n_neighbors': 4, 'weights': 'distance', 'metric': 'l1'}
[2023-12-19 04:34:59] Start trial 0.63:
                      Params: {'algorithm': 'KNeighborsRegressor', 'scale': None, 'n_neighbors': 6, 'weights': 'distance', 'metric': 'l1'}
[2023-12-19 04:35:00

In [189]:
result = cross_validation(
    KNeighborsRegressor(n_neighbors=2, weights="distance", metric="cityblock"),
    n_splits=5,
    scale="Standard",
)


print(f"Best trial results" f"\nMSE: {result}")

Best trial results
MSE: 105.61941520641236
